# 10 Classification Fine-Tuning

## Purpose

This notebook fine-tunes one multi-label sequence classifier for the downstream glycan subtype task.

## Why this notebook matters

Notebook 09 prepares the classification tables, but those saved CSV files still need to be turned into tokenized model inputs, trained with a selected classifier run mode, and reviewed on the validation split before the test set is touched. This notebook handles that training-and-review workflow and saves the classifier artifacts that notebook 11 will use for final evaluation.

## Inputs

- one saved MLM `best_model/` folder for fresh classifier runs, or one saved classifier checkpoint folder for resume or continuation runs
- `train_classification.csv`
- `val_classification.csv`
- `test_classification.csv`
- `label_vocabulary.csv`

## Outputs

- classifier checkpoints in `checkpoints/classification/<tokenizer_family>/<experiment_name>/<classifier_run_label>/`
- validation-review artifacts in `results/classification_finetuning/<tokenizer_family>/<experiment_name>/<classifier_run_label>/`
- `training_config.json`
- `trainer_state.json`
- `loss_history.csv`
- `loss_curves.png`
- `validation_review_summary.json`
- `validation_metrics.csv`
- `validation_threshold_scan.csv`
- `validation_prediction_table.csv`
- `best_threshold.json`


## User settings

Review this cell before running the notebook. All notebook-specific values that may need editing are collected here so the rest of the notebook can stay focused on the workflow.

**Run modes**
- `fresh_mlm_checkpoint`: start a new classifier from an MLM `best_model/` folder
- `fresh_random_init`: start a new classifier with random weights but the same tokenizer/config family
- `resume_checkpoint`: resume an interrupted classifier run from a saved `checkpoint-*` folder
- `continue_best_model`: start a new classifier run from a saved classifier `best_model/` folder

**Settings to review**
- `PROJECT_ROOT`: root project folder in Google Drive
- `CLASSIFICATION_PREP_RESULTS_DIRNAME`: notebook-09 results folder to read from
- `RUN_MODE`, `PRETRAINED_MODEL_DIR`, and `RESUME_SOURCE_DIR`: choose how this classifier run should start
- `CLASSIFIER_RUN_STEM`: readable stem used when the notebook creates a new classifier run label
- `OVERWRITE_EXISTING_OUTPUTS`: whether notebook-10 outputs may replace existing files
- the training hyperparameters and threshold grid

**Expected output**
- a printed summary of the active run settings


In [ ]:
from pathlib import Path

# Update this path if your Google Drive project folder uses a different name or location.
PROJECT_ROOT = Path('/content/drive/MyDrive/ProjectRoot')

# Repository settings used to sync the latest notebook and helper code into Colab.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
GITHUB_REF = 'main'

# Choose which notebook-09 results folder should feed this fine-tuning run.
CLASSIFICATION_PREP_RESULTS_DIRNAME = 'classification_prep'

# Choose one classifier run mode.
RUN_MODE = 'fresh_mlm_checkpoint'

# Use PRETRAINED_MODEL_DIR for fresh classifier runs. For continuation or resume
# modes, leave this as None and point RESUME_SOURCE_DIR at the saved classifier folder.
PRETRAINED_MODEL_DIR = PROJECT_ROOT / 'checkpoints' / 'manual' / 'mlm15_L6_H512_A8_lr00001_ep100_setv1_train_only_v2' / 'best_model'
RESUME_SOURCE_DIR = None

# This stem is used only when the notebook creates a new classifier run label.
CLASSIFIER_RUN_STEM = 'cls_lr2e-5_ep10_bs16'

# Set this to True only when you intentionally want to replace a finished or
# partially finished notebook-10 output set.
OVERWRITE_EXISTING_OUTPUTS = False

# Training hyperparameters.
MAX_LENGTH = 130
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
INITIAL_NUM_TRAIN_EPOCHS = 100
CONTINUATION_NUM_TRAIN_EPOCHS = 20
BASE_LEARNING_RATE = 2e-5
CONTINUATION_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 15
SAVE_TOTAL_LIMIT = 3
SEED = 42

# Validation-threshold candidates to compare after training.
THRESHOLD_GRID = [0.30, 0.40, 0.50, 0.60]

print(f'Project root: {PROJECT_ROOT}')
print(f'GitHub repository: {GITHUB_OWNER}/{REPO_NAME} @ {GITHUB_REF}')
print(f'Classification prep results folder: {CLASSIFICATION_PREP_RESULTS_DIRNAME}')
print(f'Run mode: {RUN_MODE}')
print(f'Pretrained model dir: {PRETRAINED_MODEL_DIR}')
print(f'Resume source dir: {RESUME_SOURCE_DIR}')
print(f'Classifier run stem: {CLASSIFIER_RUN_STEM}')
print(f'Overwrite existing outputs: {OVERWRITE_EXISTING_OUTPUTS}')
print(f'Max length: {MAX_LENGTH}')
print(f'Initial epochs: {INITIAL_NUM_TRAIN_EPOCHS}')
print(f'Continuation epochs: {CONTINUATION_NUM_TRAIN_EPOCHS}')
print(f'Base learning rate: {BASE_LEARNING_RATE}')
print(f'Continuation learning rate: {CONTINUATION_LEARNING_RATE}')
print(f'Threshold grid: {THRESHOLD_GRID}')


## Runtime setup

This cell prepares the Colab runtime so the notebook can read data from Google Drive, import the latest project code from GitHub, and keep progress-bar output in a plain-text format that saves cleanly back to GitHub.

**Expected output**
- confirmation that Google Drive is mounted
- confirmation that the repository was cloned or updated
- confirmation of the active repository directory

**How to interpret the result**
- if the repository sync fails, later imports from `src` will fail or use stale code
- if the repository directory is unexpected, the notebook may not be importing the intended project version


In [ ]:
# Standard library imports are needed here because the shared src helpers are
# not available until after the repository has been cloned or updated.
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Mount Google Drive so the notebook can read prepared classification tables
# and save classifier checkpoints and validation-review outputs.
drive.mount('/content/drive')

# Clone the repository the first time the notebook runs. If it already exists,
# pull the requested branch so the notebook uses the latest helper code.
REPO_DIR = Path('/content') / REPO_NAME
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'

if not REPO_DIR.exists():
    print(f'Cloning repository from {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'Repository already exists at {REPO_DIR}.')

print(f"Updating repository to the latest '{GITHUB_REF}' changes...")
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_REF],
    check=True,
)

# Add the repository root to the import path so later cells can load shared
# helpers from the src package.
repo_dir_str = str(REPO_DIR)
if repo_dir_str not in sys.path:
    sys.path.insert(0, repo_dir_str)

# Reuse the plain-text tqdm patch so notebook output stays readable after it
# is saved back from Colab.
from src.pretraining import patch_transformers_tqdm_for_plain_text

patch_transformers_tqdm_for_plain_text()

print(f'Repository directory: {REPO_DIR}')


## Validate the run settings and build the run context

This cell hands the user settings to the shared classifier-training helper. The helper validates the notebook-09 input files, resolves the effective learning rate and epoch count for the chosen run mode, derives the tokenizer family and experiment name, builds the standard checkpoint and results folders, and checks whether the planned notebook-10 outputs may be written safely.

The same cell also saves a `training_config.json` snapshot so the run settings are recorded before training begins.

**Expected output**
- a printed summary of the resolved classifier run label, tokenizer family, experiment name, and output folders
- the saved `training_config.json` path

**How to interpret the result**
- if this cell fails, the problem is usually a wrong input path, an invalid run-mode combination, or an overwrite-protection block
- if the resolved run label or output folders look wrong, stop here and correct the user settings before training


In [ ]:
from IPython.display import Image, display
import pandas as pd
from transformers import EarlyStoppingCallback, Trainer

from src.classification_training import (
    build_classification_prediction_table,
    build_classification_training_arguments,
    build_classification_validation_review,
    build_classification_datasets,
    build_hf_compute_metrics,
    build_label_name_to_id,
    binarize_multilabel_predictions,
    load_classification_tables,
    load_classification_tokenizer,
    load_sequence_classification_model,
    prepare_classification_run,
    save_classification_validation_review,
    save_json,
    save_threshold_scan,
    scan_global_thresholds,
    sigmoid_predictions_from_logits,
)
from src.training_diagnostics import save_loss_curve_plot

# Build the validated run context that later cells will reuse for prepared
# input files, classifier checkpoint paths, learning-rate selection, and
# resume behavior.
run_context = prepare_classification_run(
    project_root=PROJECT_ROOT,
    classification_prep_dirname=CLASSIFICATION_PREP_RESULTS_DIRNAME,
    run_mode=RUN_MODE,
    pretrained_model_dir=PRETRAINED_MODEL_DIR,
    classifier_run_stem=CLASSIFIER_RUN_STEM,
    resume_source_dir=RESUME_SOURCE_DIR,
    overwrite_existing_outputs=OVERWRITE_EXISTING_OUTPUTS,
    initial_num_train_epochs=INITIAL_NUM_TRAIN_EPOCHS,
    continuation_num_train_epochs=CONTINUATION_NUM_TRAIN_EPOCHS,
    base_learning_rate=BASE_LEARNING_RATE,
    continuation_learning_rate=CONTINUATION_LEARNING_RATE,
    random_seed=SEED,
)

run_settings = run_context['settings']
prep_input_paths = run_context['prep_input_paths']
output_paths = run_context['output_paths']

# Save the resolved training settings now so the run folder records exactly
# which configuration was intended for this classifier training job.
save_json(run_settings, output_paths['training_config_path'])

print(f"Run mode: {run_settings['run_mode']}")
print(f"Model source dir: {run_settings['model_source_dir']}")
print(f"Classifier run label: {run_settings['classifier_run_label']}")
print(f"Tokenizer family: {run_settings['tokenizer_family']}")
print(f"Pretrain experiment name: {run_settings['pretrain_experiment_name']}")
print(f"Resolved learning rate: {run_settings['learning_rate']}")
print(f"Resolved training epochs: {run_settings['num_train_epochs']}")
print(f"Resolved random seed: {run_settings['seed']}")
print(f"Classification prep directory: {prep_input_paths['classification_prep_dir']}")
print(f"Results directory: {output_paths['results_dir']}")
print(f"Checkpoint directory: {output_paths['checkpoint_dir']}")
print(f"Training config path: {output_paths['training_config_path']}")

if run_context['resume_from_checkpoint']:
    print(f"Resume checkpoint: {run_context['resume_from_checkpoint']}")


## Load the prepared classification tables

This cell reads the notebook-09 outputs that will feed the classifier run.

We inspect the split sizes and label vocabulary here so obvious data issues are caught before any tokenization or training work begins.

**Expected output**
- printed train, validation, and test row counts
- the number of subtype labels in the saved vocabulary
- a preview of the label-vocabulary table

**How to interpret the result**
- very small split sizes usually indicate that notebook 09 saved different data than expected
- the label-vocabulary preview is a quick way to confirm that the saved label IDs and names look consistent with the prepared dataset


In [ ]:
# Load the prepared notebook-09 split tables and reconstruct the label-name
# to label-ID mapping that the classifier will use.
classification_tables = load_classification_tables(
    train_csv_path=prep_input_paths['train_classification_path'],
    val_csv_path=prep_input_paths['val_classification_path'],
    test_csv_path=prep_input_paths['test_classification_path'],
    label_vocabulary_path=prep_input_paths['label_vocabulary_path'],
)

train_df = classification_tables['train_df']
val_df = classification_tables['val_df']
test_df = classification_tables['test_df']
label_vocabulary_df = classification_tables['label_vocabulary_df']
label_name_to_id = build_label_name_to_id(label_vocabulary_df)

# Save the exact label vocabulary used for this classifier run so later
# notebooks do not depend on an external file that might change.
label_vocabulary_df.to_csv(output_paths['label_vocabulary_snapshot_path'], index=False)

print(f'Train rows: {len(train_df):,}')
print(f'Validation rows: {len(val_df):,}')
print(f'Test rows: {len(test_df):,}')
print(f'Number of subtype labels: {len(label_name_to_id):,}')
print(f"Label vocabulary snapshot path: {output_paths['label_vocabulary_snapshot_path']}")

display(label_vocabulary_df.head(10))


## Tokenize the classification datasets

This cell converts the prepared classification rows into tokenized model inputs and multi-hot label vectors.

The tokenized datasets must be built before the Hugging Face `Trainer` can run, and the printed sizes provide a quick sanity check that tokenization kept the row counts aligned.

**Expected output**
- printed train, validation, and test dataset sizes
- the tokenizer max-length setting used for truncation and padding

**How to interpret the result**
- dataset sizes should match the corresponding prepared table sizes
- if the counts disagree, something went wrong during target encoding or tokenization


In [ ]:
# Load the tokenizer that matches the selected starting checkpoint and use it
# to build tokenized train, validation, and test datasets.
tokenizer = load_classification_tokenizer(run_context['tokenizer_source_dir'])
dataset_bundle = build_classification_datasets(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    tokenizer=tokenizer,
    label_name_to_id=label_name_to_id,
    max_length=MAX_LENGTH,
)

train_dataset = dataset_bundle['train_dataset']
val_dataset = dataset_bundle['val_dataset']
test_dataset = dataset_bundle['test_dataset']

print(f'Train dataset rows: {len(train_dataset):,}')
print(f'Validation dataset rows: {len(val_dataset):,}')
print(f'Test dataset rows: {len(test_dataset):,}')
print(f'Tokenizer max length: {MAX_LENGTH}')


## Build the classifier model and training arguments

This cell initializes the classifier model and the Hugging Face training arguments for the selected run.

Fresh runs either reuse the MLM encoder weights or start from random weights, while resume or continuation modes start from a saved classifier checkpoint. The printed summary confirms the runtime device, effective learning rate, epoch count, and checkpoint folder before training begins.

**Expected output**
- the runtime device name
- the model-load mode used for this run
- the effective learning rate and epoch count
- the classifier checkpoint directory

**How to interpret the result**
- if the model-load mode does not match the intended run mode, the user settings should be corrected before training
- the checkpoint directory should match the classifier run label printed earlier


In [ ]:
import torch

runtime_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Build the classifier from the selected starting source.
model = load_sequence_classification_model(
    pretrained_model_dir=run_context['model_source_dir'],
    num_labels=len(label_name_to_id),
    initialization_mode=run_context['model_load_mode'],
    device=runtime_device,
)

# Build the shared Hugging Face TrainingArguments bundle so output paths,
# checkpointing, and random seeding stay consistent across reruns.
training_bundle = build_classification_training_arguments(
    checkpoint_dir=output_paths['checkpoint_dir'],
    learning_rate=run_settings['learning_rate'],
    train_batch_size=TRAIN_BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=run_settings['num_train_epochs'],
    weight_decay=WEIGHT_DECAY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    random_seed=run_settings['seed'],
)
training_args = training_bundle['training_args']

# Build the Trainer that will handle optimization, evaluation, and checkpoint
# saving. Metric reporting uses a neutral 0.50 threshold during training;
# detailed threshold comparison happens later on the validation split.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=build_hf_compute_metrics(threshold=0.50),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(f'Runtime device: {runtime_device}')
print(f"Model load mode: {run_context['model_load_mode']}")
print(f"Effective learning rate: {run_settings['learning_rate']}")
print(f"Effective training epochs: {run_settings['num_train_epochs']}")
print(f"fp16 enabled: {training_bundle['fp16_enabled']}")
print(f"Classifier checkpoint output dir: {output_paths['checkpoint_dir']}")


## Train the classifier

This is the main training step.

At this stage the notebook should only use the training and validation splits. If the selected run mode is `resume_checkpoint`, the cell resumes the interrupted Hugging Face trainer state from the saved checkpoint folder.

**Expected output**
- training progress messages from Hugging Face Trainer
- confirmation that the best-model export and validation metrics were saved

**How to interpret the result**
- a successful run should end with a saved `best_model/` folder and a one-row validation-metrics CSV
- if a resume run starts from the wrong checkpoint, stop and confirm the `RESUME_SOURCE_DIR` setting


In [ ]:
import shutil
import torch

# Resume interrupted training only when the selected run mode supplied a
# checkpoint directory. Fresh and continue-best-model runs start normally.
train_kwargs = {}
if run_context['resume_from_checkpoint'] is not None:
    print(f"Resuming trainer state from checkpoint: {run_context['resume_from_checkpoint']}")
    train_kwargs['resume_from_checkpoint'] = str(run_context['resume_from_checkpoint'])

trainer.train(**train_kwargs)

# Save a clean exported best-model folder in addition to the Trainer-managed
# checkpoint folders so notebook 11 has a stable path to load.
trainer.save_model(output_paths['best_model_dir'])
tokenizer.save_pretrained(output_paths['best_model_dir'])
trainer.save_state()

# Copy the Trainer state into the results folder so the validation-review
# outputs stay easy to find alongside the loss curves and threshold scan.
trainer_state_path = output_paths['checkpoint_dir'] / 'trainer_state.json'
if trainer_state_path.exists():
    shutil.copy2(trainer_state_path, output_paths['trainer_state_copy_path'])

# Save one explicit validation-metrics row after training completes so the run
# folder contains a compact summary of the final validation evaluation.
validation_metrics = trainer.evaluate(eval_dataset=val_dataset)
pd.DataFrame([validation_metrics]).to_csv(output_paths['validation_metrics_path'], index=False)

print('Training complete.')
print(f"Best-model export: {output_paths['best_model_dir']}")
print(f"Trainer state copy: {output_paths['trainer_state_copy_path']}")
print(f"Validation metrics path: {output_paths['validation_metrics_path']}")


## Review training and validation loss

This cell applies the same basic validation-review framework used in notebook 05, but now to the classifier fine-tuning run.

It loads the saved trainer history, separates training and validation loss rows, saves a merged `loss_history.csv`, saves the loss-curve plot, and builds a compact `validation_review_summary.json` that includes the best validation epoch and a simple continuation recommendation.

**Expected output**
- a preview of the merged loss-history table
- a small summary table containing the best epoch and continuation recommendation
- the saved loss-curve image displayed inline

**How to interpret the result**
- smooth validation-loss improvement is a good sign that the classifier fit is stable
- a continuation recommendation of "worth considering" means the best validation epoch happened late enough that a longer run may still help


In [ ]:
# Build the classifier validation-review bundle from the saved Trainer state.
validation_review_bundle = build_classification_validation_review(
    trainer_state_path=output_paths['trainer_state_copy_path'],
    tokenizer_family=run_settings['tokenizer_family'],
    pretrain_experiment_name=run_settings['pretrain_experiment_name'],
    classifier_run_label=run_settings['classifier_run_label'],
    run_mode=run_settings['run_mode'],
    total_epochs=run_settings['num_train_epochs'],
)

# Save the merged loss history and the compact validation-review summary.
saved_review_paths = save_classification_validation_review(
    validation_review_bundle=validation_review_bundle,
    output_paths=output_paths,
)

# Save the loss-curve plot that matches the merged history table.
save_loss_curve_plot(
    validation_review_bundle['train_rows'],
    validation_review_bundle['eval_rows'],
    output_paths['loss_curve_path'],
)

display(validation_review_bundle['loss_history_df'].head())
display(validation_review_bundle['summary_df'])
display(Image(filename=str(output_paths['loss_curve_path'])))

print(f"Loss history path: {saved_review_paths['loss_history_path']}")
print(f"Validation review summary path: {saved_review_paths['validation_review_summary_path']}")
print(f"Loss curve path: {output_paths['loss_curve_path']}")


## Scan a few validation thresholds

The classifier is trained now, so this cell compares a small set of global probability thresholds on the validation split.

The goal is to keep threshold selection strictly on the validation set and save a ranked comparison table before notebook 11 ever looks at the test set.

**Expected output**
- a threshold-comparison table sorted by validation performance
- a saved CSV copy of the scan and a saved JSON file containing the top threshold row

**How to interpret the result**
- the first row is the current best global threshold according to the saved sort order
- large performance differences across thresholds suggest that threshold choice matters a lot for this classifier run


In [ ]:
# Predict on the validation split only so the threshold scan stays part of
# the model-tuning workflow rather than leaking test information.
val_prediction_output = trainer.predict(val_dataset)
val_logits = val_prediction_output.predictions
if isinstance(val_logits, tuple):
    val_logits = val_logits[0]

val_probabilities = sigmoid_predictions_from_logits(val_logits)
val_true_labels = val_prediction_output.label_ids

threshold_results_df = scan_global_thresholds(
    probabilities=val_probabilities,
    true_labels=val_true_labels,
    thresholds=THRESHOLD_GRID,
)
save_threshold_scan(threshold_results_df, output_paths['validation_threshold_scan_path'])

best_threshold_row = threshold_results_df.iloc[0].to_dict()
save_json(best_threshold_row, output_paths['best_threshold_path'])

display(threshold_results_df)
print(f"Threshold scan path: {output_paths['validation_threshold_scan_path']}")
print(f"Best-threshold JSON path: {output_paths['best_threshold_path']}")


## Save a human-readable validation prediction table

This cell creates a readable validation table that lines each glycan back up with its true labels, predicted labels, and top probabilities.

This table is useful for later manual review because it is much easier to inspect than a raw logits matrix.

**Expected output**
- a preview of the saved validation prediction table
- confirmation of the chosen threshold and saved CSV path

**How to interpret the result**
- `predicted_labels_json` shows the labels called positive at the chosen validation threshold
- the top-probability preview helps identify near-miss labels and overconfident mistakes


In [ ]:
# Convert the selected validation threshold into binary predictions and save a
# readable table for later inspection.
chosen_threshold = float(best_threshold_row['threshold'])
val_binary_predictions = binarize_multilabel_predictions(
    val_probabilities,
    threshold=chosen_threshold,
)

validation_prediction_table_df = build_classification_prediction_table(
    source_df=dataset_bundle['val_df'].assign(split='val'),
    probabilities=val_probabilities,
    predicted_labels=val_binary_predictions,
    label_vocabulary_df=label_vocabulary_df,
)
validation_prediction_table_df.to_csv(output_paths['validation_prediction_table_path'], index=False)

print(f'Chosen validation threshold: {chosen_threshold:.2f}')
print(f"Validation prediction table path: {output_paths['validation_prediction_table_path']}")
display(validation_prediction_table_df.head(10))


## Next step

If the validation-loss review, threshold scan, and prediction table all look reasonable, the next notebook should be the final classifier evaluation notebook.

Notebook 11 should:
- load the saved classifier from `best_model/`
- load the saved `best_threshold.json`
- run final predictions on the test set once
- save the final test metrics and test prediction table

That keeps the test set as a final reporting step instead of a tuning loop.
